# Leakage-aware pedestrian detector experiment

Train augmented YOLOv8n and YOLOv8s, keep Umair's external images out of training, evaluate only after label review, and export ONNX weights.

In [ ]:
!pip -q install ultralytics onnx onnxruntime onnxslim
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
from ultralytics import YOLO
from pathlib import Path
import shutil
ROOT = Path('/content/drive/MyDrive/ADAS_Project')
DATA = Path('/content/dataset')
EXTERNAL = Path('/content/external_test')
RUNS = ROOT / 'runs'
DATA_YAML = Path('/content/data.yaml')

In [ ]:
!unzip -q -o '/content/drive/MyDrive/ADAS_Project/dataset.zip' -d /content/
external_zip = ROOT / 'Test Data.zip'
if external_zip.exists():
    EXTERNAL.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(external_zip), str(EXTERNAL))
DATA_YAML.write_text("path: /content/dataset\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 1\nnames: ['pedestrian']\n")
print('dataset extracted')

In [ ]:
# Generate pedestrian labels in Colab, where PyTorch can use the GPU.
labeler = YOLO('yolov8m.pt')
for split in ('train', 'val', 'test'):
    image_dir = DATA / 'images' / split
    label_dir = DATA / 'labels' / split
    for image_path in image_dir.glob('*'):
        if not image_path.name.startswith(('pedestrian', 'pedestrian_rotated')):
            continue
        result = labeler.predict(source=str(image_path), conf=0.30, verbose=False)[0]
        lines = []
        for box in result.boxes:
            if int(box.cls[0]) == 0:
                x, y, w, h = box.xywhn[0].tolist()
                lines.append(f'0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n')
        (label_dir / f'{image_path.stem}.txt').write_text(''.join(lines))
print('Pedestrian labels generated for train, val, and test.')

In [ ]:
augmentation = dict(degrees=10.0, translate=0.1, scale=0.5, fliplr=0.5, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, mosaic=1.0)
YOLO('yolov8n.pt').train(data=str(DATA_YAML), epochs=50, imgsz=640, batch=16, project=str(RUNS), name='yolov8n_augmented', **augmentation)

In [ ]:
YOLO('yolov8s.pt').train(data=str(DATA_YAML), epochs=50, imgsz=640, batch=16, project=str(RUNS), name='yolov8s_augmented', **augmentation)

In [ ]:
for run_name in ('yolov8n_augmented', 'yolov8s_augmented'):
    detector = YOLO(str(RUNS / run_name / 'weights' / 'best.pt'))
    metrics = detector.val(data=str(DATA_YAML), split='test')
    onnx_path = detector.export(format='onnx', dynamic=True)
    print(run_name, float(metrics.box.map50), float(metrics.box.map), onnx_path)

External evaluation requires reviewed labels. Use the external images only for final testing, report those results separately from the original test split, and download `results.png`, `confusion_matrix.png`, `F1_curve.png`, `best.pt`, and `best.onnx` from each run folder.